# EMMA — Edge Multimodal Inference Pipeline

**Pipeline:**
1. **CLIP Encoding** — frozen CLIP encodes image+text pairs → 512-dim embeddings (cached to Drive)
2. **AutoEncoder Compression** — compresses 512→64 dims (8× bandwidth reduction)
3. **Server Training** — projection MLP + GPT-2 + match head trained on 64-dim latents
4. **Evaluation** — test accuracy + plots

Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
!pip install transformers datasets -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/emma_checkpoints'
EMBED_CACHE    = os.path.join(CHECKPOINT_DIR, 'embed_cache')
LATENT_CACHE   = os.path.join(EMBED_CACHE,    'ae_latents')
COCO_CACHE     = '/content/coco_cache'

for p in [CHECKPOINT_DIR, EMBED_CACHE, LATENT_CACHE, COCO_CACHE]:
    os.makedirs(p, exist_ok=True)

print('Drive mounted.')
print('Embed cache :', EMBED_CACHE)
print('Latent cache:', LATENT_CACHE)

In [ ]:
import os, pickle, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from transformers import (
    CLIPModel, CLIPTokenizer, CLIPImageProcessor,
    AutoModelForCausalLM, AutoTokenizer
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## Hyperparameters

In [ ]:
# Data
N_TRAIN      = 10_000
N_VALID      =  1_000
N_TEST       =  1_000
MAX_TEXT_LEN = 77          # CLIP tokenizer limit
BATCH_SIZE   = 256
HF_DATASET   = 'clip-benchmark/wds_mscoco_captions'

# AutoEncoder
AE_EPOCHS    = 50
AE_LR        = 1e-4
D_SHARED     = 512         # CLIP ViT-B/32 joint embedding dim
D_LATENT     = 64          # compressed dim  (8x compression)

# Server
SERVER_EPOCHS  = 30
SERVER_LR      = 1e-4
N_SOFT_TOKENS  = 8
INSTRUCTION    = 'Does the image match the description?'
GRAD_CLIP      = 1.0
PATIENCE       = 10
MIN_DELTA      = 1e-4

print(f'Compression: {D_SHARED}→{D_LATENT} ({D_SHARED//D_LATENT}x)')

## Model Definitions

In [ ]:
# ── CLIP Encoders (frozen) ────────────────────────────────────────────────────
class TextEncoder(nn.Module):
    """CLIP text encoder → 512-dim joint embeddings."""
    def __init__(self, model_name='openai/clip-vit-base-patch32'):
        super().__init__()
        clip = CLIPModel.from_pretrained(model_name)
        self.text_model      = clip.text_model
        self.text_projection = clip.text_projection
        for p in self.parameters(): p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        return self.text_projection(out.pooler_output)   # [B, 512]


class ImageEncoder(nn.Module):
    """CLIP vision encoder → 512-dim joint embeddings."""
    def __init__(self, model_name='openai/clip-vit-base-patch32'):
        super().__init__()
        clip = CLIPModel.from_pretrained(model_name)
        self.vision_model      = clip.vision_model
        self.visual_projection = clip.visual_projection
        for p in self.parameters(): p.requires_grad = False

    def forward(self, pixel_values):
        out = self.vision_model(pixel_values=pixel_values)
        return self.visual_projection(out.pooler_output)  # [B, 512]


class EdgePipeline(nn.Module):
    """Frozen CLIP edge pipeline — encoding only, no training."""
    def __init__(self):
        super().__init__()
        self.text_encoder  = TextEncoder()
        self.image_encoder = ImageEncoder()

    def forward(self, text_inputs, image_inputs):
        t = self.text_encoder(**text_inputs)
        v = self.image_encoder(image_inputs['pixel_values'])
        return (t + v) / 2     # mean fusion → [B, 512]


# ── AutoEncoder (compression) ─────────────────────────────────────────────────
class AEEncoder(nn.Module):
    """Edge-side: 512 → 64 dims."""
    def __init__(self, d_in=D_SHARED, d_latent=D_LATENT,
                 hidden=(512, 256, 128)):
        super().__init__()
        layers, dim = [], d_in
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.ReLU(inplace=True)]; dim = h
        layers.append(nn.Linear(dim, d_latent))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


class AEDecoder(nn.Module):
    """Server-side: 64 → 512 dims."""
    def __init__(self, d_latent=D_LATENT, d_out=D_SHARED,
                 hidden=(128, 256, 512)):
        super().__init__()
        layers, dim = [], d_latent
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.ReLU(inplace=True)]; dim = h
        layers.append(nn.Linear(dim, d_out))
        self.net = nn.Sequential(*layers)

    def forward(self, z): return self.net(z)


class AutoEncoder(nn.Module):
    """Full AE used for training. Pure MSE loss, no KL term."""
    def __init__(self, d_in=D_SHARED, d_latent=D_LATENT):
        super().__init__()
        self.encoder = AEEncoder(d_in, d_latent)
        self.decoder = AEDecoder(d_latent, d_in)

    def forward(self, x): return self.decoder(self.encoder(x))


# ── Server Pipeline ───────────────────────────────────────────────────────────
class ServerPipeline(nn.Module):
    """Projection MLP + frozen GPT-2 + binary match head."""
    def __init__(self, llm, d_latent=D_LATENT, n_soft=N_SOFT_TOKENS):
        super().__init__()
        self.llm       = llm
        self.n_soft    = n_soft
        d_llm          = llm.config.hidden_size
        for p in self.llm.parameters(): p.requires_grad = False
        # Trainable layers
        self.projection = nn.Sequential(
            nn.Linear(d_latent, 256), nn.ReLU(),
            nn.Linear(256, n_soft * d_llm),
        )
        self.match_head = nn.Linear(d_llm, 1)

    def forward(self, latent, instr_ids, instr_mask):
        B     = latent.size(0)
        d_llm = self.llm.config.hidden_size
        soft  = self.projection(latent).view(B, self.n_soft, d_llm)
        instr_emb  = self.llm.transformer.wte(instr_ids)
        inputs_emb = torch.cat([soft, instr_emb], dim=1)
        out = self.llm.transformer(inputs_embeds=inputs_emb)
        cls = out.last_hidden_state[:, 0, :]
        return self.match_head(cls)   # [B, 1]

print('All model classes defined.')

## Data Loading
Only needed if CLIP embedding cache is not on Drive. Skipped automatically otherwise.

In [ ]:
embed_exists = os.path.exists(os.path.join(EMBED_CACHE, 'train_fused.pt'))

if embed_exists:
    print('Embed cache found on Drive — skipping COCO loading.')
else:
    print('No cache found. Loading COCO for CLIP encoding...')

    def _stream_samples(n, offset=0):
        f = os.path.join(COCO_CACHE, f'samples_{offset}_{n}.pkl')
        if os.path.exists(f):
            with open(f, 'rb') as fp: return pickle.load(fp)
        from datasets import load_dataset
        print(f'  streaming {n} images (offset={offset})...')
        ds = load_dataset(HF_DATASET, split='train', streaming=True)
        out = []
        for idx, item in enumerate(ds):
            if idx < offset: continue
            if len(out) >= n: break
            txt      = item['txt']
            captions = [c.strip() for c in txt.splitlines() if c.strip()] if isinstance(txt, str) else [str(txt).strip()]
            if captions and item['jpg'] is not None:
                img = item['jpg']
                if img.size != (224,224): img = img.resize((224,224))
                out.append({'image': img, 'captions': captions})
        with open(f, 'wb') as fp: pickle.dump(out, fp)
        return out

    class COCODataset(Dataset):
        """Lazy — images processed on-the-fly to avoid RAM spike."""
        def __init__(self, raw, seed=0):
            self.proc = CLIPImageProcessor.from_pretrained('openai/clip-vit-base-patch32')
            self.tok  = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
            rng = random.Random(seed)
            self.images   = [s['image']               for s in raw]
            self.captions = [rng.choice(s['captions']) for s in raw]
            N   = len(self.images)
            neg = list(range(N)); rng.shuffle(neg)
            self.pairs = ([(i,i,1) for i in range(N)] +
                          [(i,(neg[i]+1)%N if neg[i]==i else neg[i],0) for i in range(N)])
            rng.shuffle(self.pairs)

        def __len__(self): return len(self.pairs)

        def __getitem__(self, idx):
            img_i, cap_i, label = self.pairs[idx]
            pv  = self.proc(images=self.images[img_i], return_tensors='pt')['pixel_values'][0]
            enc = self.tok(self.captions[cap_i], max_length=MAX_TEXT_LEN,
                           padding='max_length', truncation=True, return_tensors='pt')
            return {'pixel_values':   pv,
                    'input_ids':      enc['input_ids'][0],
                    'attention_mask': enc['attention_mask'][0],
                    'label':          torch.tensor(float(label))}

    offsets  = {'train': 0, 'valid': N_TRAIN, 'test': N_TRAIN+N_VALID}
    sizes    = {'train': N_TRAIN, 'valid': N_VALID, 'test': N_TEST}
    raw_data = {s: _stream_samples(sizes[s], offsets[s]) for s in ('train','valid','test')}
    datasets = {s: COCODataset(raw_data[s], seed={'train':0,'valid':1,'test':2}[s])
                for s in raw_data}
    loaders  = {s: DataLoader(datasets[s], batch_size=64, shuffle=(s=='train'),
                               num_workers=0) for s in datasets}
    print({s: len(datasets[s]) for s in datasets})

## Stage 1 — CLIP Encoding
Runs once and saves to Drive. Auto-skipped if cache exists.

In [ ]:
if embed_exists:
    print('Cache exists — skipping CLIP encoding.')
    for split in ('train','valid','test'):
        path = os.path.join(EMBED_CACHE, f'{split}_fused.pt')
        mb   = os.path.getsize(path)/1e6
        print(f'  {split}_fused.pt  {mb:.1f} MB')
else:
    print('Encoding all splits with frozen CLIP...')
    edge = EdgePipeline().to(DEVICE)
    edge.eval()
    for split, loader in loaders.items():
        fused_list, label_list = [], []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                fused = edge(
                    {'input_ids': batch['input_ids'], 'attention_mask': batch['attention_mask']},
                    {'pixel_values': batch['pixel_values']}
                )
                fused_list.append(fused.cpu())
                label_list.append(batch['label'].cpu())
        torch.save(torch.cat(fused_list), os.path.join(EMBED_CACHE, f'{split}_fused.pt'))
        torch.save(torch.cat(label_list), os.path.join(EMBED_CACHE, f'{split}_label.pt'))
        n = sum(len(f) for f in fused_list)
        print(f'  {split}: {n} samples cached')
    del edge; torch.cuda.empty_cache()
    print('Done. Embeddings saved to Drive.')

## Stage 2 — AutoEncoder Compression

In [ ]:
# Load cached CLIP embeddings
train_fused = torch.load(os.path.join(EMBED_CACHE, 'train_fused.pt'), weights_only=True)
valid_fused = torch.load(os.path.join(EMBED_CACHE, 'valid_fused.pt'), weights_only=True)
print(f'Train: {train_fused.shape}  Valid: {valid_fused.shape}')

ae_train_loader = DataLoader(torch.utils.data.TensorDataset(train_fused),
                              batch_size=BATCH_SIZE, shuffle=True)
ae_valid_loader = DataLoader(torch.utils.data.TensorDataset(valid_fused),
                              batch_size=BATCH_SIZE)

ae        = AutoEncoder(D_SHARED, D_LATENT).to(DEVICE)
optimizer = AdamW(ae.parameters(), lr=AE_LR, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=AE_EPOCHS * len(ae_train_loader))

best_ae_loss = float('inf')
ae_ckpt      = os.path.join(CHECKPOINT_DIR, 'ae.pt')
ae_history   = {'train_mse': [], 'val_mse': [], 'cos_sim': []}

print(f'Training AutoEncoder ({D_SHARED}→{D_LATENT} dims, {AE_EPOCHS} epochs)...')
for epoch in range(1, AE_EPOCHS + 1):
    ae.train()
    tl, nb = 0., 0
    for (x,) in ae_train_loader:
        x    = x.to(DEVICE)
        loss = F.mse_loss(ae(x), x)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(ae.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        tl += loss.item(); nb += 1

    ae.eval()
    vl, vb = 0., 0
    with torch.no_grad():
        for (x,) in ae_valid_loader:
            x = x.to(DEVICE)
            vl += F.mse_loss(ae(x), x).item(); vb += 1
        recon_v = ae(valid_fused.to(DEVICE))
        cos     = F.cosine_similarity(valid_fused.to(DEVICE), recon_v).mean().item()

    ae_history['train_mse'].append(tl/nb)
    ae_history['val_mse'].append(vl/vb)
    ae_history['cos_sim'].append(cos)

    print(f'Epoch {epoch:03d}/{AE_EPOCHS} | train MSE {tl/nb:.4f} | '
          f'val MSE {vl/vb:.4f} | cos_sim {cos:.4f}')

    if vl/vb < best_ae_loss:
        best_ae_loss = vl/vb
        torch.save({'state_dict': ae.state_dict(),
                    'd_in': D_SHARED, 'd_latent': D_LATENT}, ae_ckpt)
        print(f'  ✓ saved')

print(f'\nBest val MSE: {best_ae_loss:.4f}')

In [ ]:
# Load best AE and encode all splits to latents
ckpt = torch.load(ae_ckpt, weights_only=False)
ae.load_state_dict(ckpt['state_dict'])
ae.eval()

print('Caching AE latents...')
for split in ('train', 'valid', 'test'):
    fused  = torch.load(os.path.join(EMBED_CACHE, f'{split}_fused.pt'), weights_only=True)
    labels = torch.load(os.path.join(EMBED_CACHE, f'{split}_label.pt'), weights_only=True)
    with torch.no_grad():
        latents = ae.encoder(fused.to(DEVICE)).cpu()
        recon   = ae.decoder(latents.to(DEVICE))
        cos     = F.cosine_similarity(fused.to(DEVICE), recon).mean().item()
    torch.save(latents, os.path.join(LATENT_CACHE, f'{split}_fused.pt'))
    torch.save(labels,  os.path.join(LATENT_CACHE, f'{split}_label.pt'))
    print(f'  {split}: {latents.shape}  cos_sim={cos:.4f}')

print('Latents saved to Drive.')

## Stage 3 — Server Training

In [ ]:
# Cached latent dataset
class CachedDataset(Dataset):
    def __init__(self, path, split):
        self.fused  = torch.load(os.path.join(path, f'{split}_fused.pt'), weights_only=True)
        self.labels = torch.load(os.path.join(path, f'{split}_label.pt'), weights_only=True)
    def __len__(self): return len(self.fused)
    def __getitem__(self, idx):
        return {'fused': self.fused[idx], 'label': self.labels[idx]}

server_loaders = {s: DataLoader(CachedDataset(LATENT_CACHE, s),
                                 batch_size=BATCH_SIZE, shuffle=(s=='train'),
                                 num_workers=0)
                  for s in ('train', 'valid', 'test')}
print({s: len(server_loaders[s].dataset) for s in server_loaders})

# Build server
llm        = AutoModelForCausalLM.from_pretrained('gpt2').to(DEVICE)
tokenizer  = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

server     = ServerPipeline(llm, d_latent=D_LATENT).to(DEVICE)
trainable  = [p for p in server.parameters() if p.requires_grad]
print(f'Trainable params: {sum(p.numel() for p in trainable):,}')

instr_enc  = tokenizer(INSTRUCTION, return_tensors='pt',
                        padding=True, truncation=True, max_length=32)
instr_ids  = instr_enc['input_ids'].to(DEVICE)
instr_mask = instr_enc['attention_mask'].to(DEVICE)

optimizer  = AdamW(trainable, lr=SERVER_LR, weight_decay=1e-2)
scheduler  = CosineAnnealingLR(optimizer, T_max=SERVER_EPOCHS * len(server_loaders['train']))

best_val_loss    = float('inf')
patience_counter = 0
server_ckpt      = os.path.join(CHECKPOINT_DIR, 'best_server.pt')
history          = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

print(f'\nTraining server ({SERVER_EPOCHS} epochs, patience={PATIENCE})...')
for epoch in range(1, SERVER_EPOCHS + 1):
    # Train
    server.train()
    tl, tc, nb = 0., 0., 0
    for batch in server_loaders['train']:
        fused = batch['fused'].to(DEVICE)
        label = batch['label'].to(DEVICE)
        B     = fused.size(0)
        logit = server(fused, instr_ids.expand(B,-1), instr_mask.expand(B,-1))
        loss  = F.binary_cross_entropy_with_logits(logit.squeeze(-1), label)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
        optimizer.step(); scheduler.step()
        tl += loss.item()
        tc += ((logit.squeeze(-1) > 0) == (label > 0.5)).float().mean().item()
        nb += 1

    # Validate
    server.eval()
    vl, vc, vb = 0., 0., 0
    with torch.no_grad():
        for batch in server_loaders['valid']:
            fused = batch['fused'].to(DEVICE)
            label = batch['label'].to(DEVICE)
            B     = fused.size(0)
            logit = server(fused, instr_ids.expand(B,-1), instr_mask.expand(B,-1))
            vl += F.binary_cross_entropy_with_logits(logit.squeeze(-1), label).item()
            vc += ((logit.squeeze(-1) > 0) == (label > 0.5)).float().mean().item()
            vb += 1

    val_loss = vl/vb
    print(f'Epoch {epoch:02d}/{SERVER_EPOCHS} | '
          f'train loss {tl/nb:.4f} acc {tc/nb:.3f} | '
          f'val loss {val_loss:.4f} acc {vc/vb:.3f}')

    history['train_loss'].append(tl/nb); history['train_acc'].append(tc/nb)
    history['val_loss'].append(val_loss); history['val_acc'].append(vc/vb)

    if best_val_loss - val_loss > MIN_DELTA:
        best_val_loss = val_loss; patience_counter = 0
        torch.save({'epoch': epoch, 'server': server.state_dict(),
                    'val_loss': best_val_loss, 'val_acc': vc/vb}, server_ckpt)
        print(f'  ✓ saved (val_acc={vc/vb:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}'); break

print(f'\nBest val loss: {best_val_loss:.4f}')

## Results

In [ ]:
# Load best server and evaluate on test set
ckpt = torch.load(server_ckpt, weights_only=False)
server.load_state_dict(ckpt['server'])
server.eval()

all_logits, all_labels = [], []
with torch.no_grad():
    for batch in server_loaders['test']:
        fused = batch['fused'].to(DEVICE)
        label = batch['label'].to(DEVICE)
        B     = fused.size(0)
        logit = server(fused, instr_ids.expand(B,-1), instr_mask.expand(B,-1))
        all_logits.append(logit.cpu()); all_labels.append(label.cpu())

logits   = torch.cat(all_logits).squeeze(-1)
labels   = torch.cat(all_labels)
test_acc = ((logits > 0) == (labels > 0.5)).float().mean().item()

print('='*45)
print(f'  Compression      : {D_SHARED}→{D_LATENT} dims ({D_SHARED//D_LATENT}x)')
print(f'  AE cos_sim       : {ae_history["cos_sim"][-1]:.4f}')
print(f'  Best val accuracy: {ckpt["val_acc"]:.4f}  (epoch {ckpt["epoch"]})')
print(f'  Test accuracy    : {test_acc:.4f}')
print(f'  Random baseline  : 0.5000')
print(f'  Gain over random : +{test_acc-0.5:.4f}')
print('='*45)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# AE reconstruction
ax = axes[0]
ax.plot(ae_history['val_mse'],  label='Val MSE')
ax.plot(ae_history['train_mse'], label='Train MSE', alpha=0.7)
ax.set(xlabel='Epoch', ylabel='MSE', title='AE Reconstruction Loss')
ax.legend()

# Server loss
ax = axes[1]
ax.plot(history['train_loss'], label='Train')
ax.plot(history['val_loss'],   label='Val')
ax.axhline(0.693, color='r', linestyle='--', label='Random (ln2)')
ax.set(xlabel='Epoch', ylabel='BCE Loss', title='Server Loss')
ax.legend()

# Server accuracy
ax = axes[2]
ax.plot(history['train_acc'], label='Train')
ax.plot(history['val_acc'],   label='Val')
ax.axhline(0.5, color='r', linestyle='--', label='Random (50%)')
ax.axhline(test_acc, color='g', linestyle='--', label=f'Test ({test_acc:.3f})')
ax.set(xlabel='Epoch', ylabel='Accuracy', title='Server Accuracy')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'results.png'), dpi=150)
plt.show()
print('Plot saved to Drive.')